In [1]:
%cd ..
%pwd

/gpfs/data/schambralab/quantitativeRehabilitation/__lab_member_homes/victor/cvfm4rehab


'/gpfs/data/schambralab/quantitativeRehabilitation/__lab_member_homes/victor/cvfm4rehab'

In [2]:
import json
import subprocess
import os
import cv2
import numpy as np
import os
import cv2
import matplotlib.pyplot as plt
from math import ceil
from typing import Dict

In [3]:
def extract_frames(path_v: str, n: int, out_dir: str):
    """
    Extract n equally spaced frames from a video using OpenCV and
    save them as PNGs in "visualization/plots/frames".

    Args:
        path_v: Path to the input video file.
        n: Number of frames to extract.
    """
    os.makedirs(out_dir, exist_ok=True)

    if os.path.isdir(out_dir):
        for fname in os.listdir(out_dir):
            file_path = os.path.join(out_dir, fname)
            if os.path.isfile(file_path):
                os.remove(file_path)
    else:
        os.makedirs(out_dir, exist_ok=True)

    cap = cv2.VideoCapture(path_v)
    if not cap.isOpened():
        raise IOError(f"Cannot open video file {path_v!r}")

    # total frame count
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        raise ValueError("Couldn't determine total frame count or video is empty.")
    if n > total_frames:
        raise ValueError(f"Requested {n} frames but video has only {total_frames} frames.")

    # compute equally spaced frame indices
    indices = np.linspace(0, total_frames - 1, n, dtype=int)

    for i, idx in enumerate(indices):
        # seek to desired frame
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            print(f"Warning: could not read frame at index {idx}")
            continue

        # save as PNG
        filename = f"frame_{i:04d}.png"
        cv2.imwrite(os.path.join(out_dir, filename), frame)

    cap.release()


In [4]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.gridspec as gridspec
import os
import textwrap
import math
from typing import Dict, List

def visualize_frames_with_metadata(
    metadata: Dict[str, str],
    save_path: str,
    frames_dir: str = "visualization/plots/frames",
    anon_suffix: str = "_anonymized.png",
    fig_width: float = 15.0,
    font_size: int = 12,
    line_spacing: float = 1.2,
    side_margin: float = 0.03,
    show: bool = False
):
    """
    Visualizes 5 image frames horizontally at the bottom of a figure
    with formatted metadata displayed above them.

    Calculates the required figure height based on frame aspect ratio
    and the number of lines needed for the metadata text (including wrapping).

    Args:
        metadata: Dictionary containing string key-value pairs for metadata.
                  Values can contain newline characters ('\n').
        save_path: Path to save the generated figure.
        frames_dir: Directory containing the image frames.
        anon_suffix: Suffix of the frame image files (e.g., '_anonymized.png').
        fig_width: Desired width of the figure in inches.
        font_size: Font size for the metadata text in points.
        line_spacing: Line spacing multiplier for the metadata text.
        side_margin: Horizontal margin on each side as a fraction of figure width.
        show: If True, display the plot interactively.
    """
    # --- 1. Find and Load Frames ---
    try:
        all_files = sorted([f for f in os.listdir(frames_dir) if f.endswith(anon_suffix)])
        if len(all_files) < 5:
            raise ValueError(f"Expected at least 5 frames with suffix '{anon_suffix}' in '{frames_dir}', found {len(all_files)}.")
        frame_files = all_files[:5] # Take the first 5
        frame_paths = [os.path.join(frames_dir, fname) for fname in frame_files]
        frames = [mpimg.imread(fp) for fp in frame_paths]
    except FileNotFoundError:
        print(f"Error: Frames directory '{frames_dir}' not found.")
        return
    except Exception as e:
        print(f"Error loading frames: {e}")
        return

    # --- 2. Calculate Frame Dimensions ---
    # Assume all frames have the same dimensions, use the first one
    frame_h_px, frame_w_px, _ = frames[0].shape
    aspect_ratio = frame_h_px / frame_w_px

    # Calculate width available for frames (considering margins)
    content_width_inches = fig_width * (1 - 2 * side_margin)
    single_frame_width_inches = content_width_inches / 5

    # Calculate the height needed for the row of frames in inches
    frame_row_height_inches = single_frame_width_inches * aspect_ratio

    # --- 3. Prepare and Calculate Text Height ---
    # Estimate character width for wrapping calculation (heuristic)
    # Based on figure width, margins, and font size.
    # This estimates how many characters roughly fit in the available width.
    # Assumes ~0.6 points width per point height for average characters.
    chars_per_inch_approx = 72 / (font_size * 0.6) if font_size > 0 else 10 # points/inch / (pts * width/height)
    wrap_width_chars = int(content_width_inches * chars_per_inch_approx)
    if wrap_width_chars <= 0:
        wrap_width_chars = 50 # Fallback minimum width

    metadata_lines: List[str] = []
    for key, value in metadata.items():
        header = f"{key}: "
        # Handle potential explicit newlines within the value first
        value_parts = str(value).split('\n')
        first_part = True
        for part in value_parts:
            # Wrap each part if it's long
            wrapped_lines = textwrap.wrap(part, width=wrap_width_chars - len(header) if first_part else wrap_width_chars,
                                          subsequent_indent="  " * (len(header)//2) if first_part else "", # Indent subsequent lines of value
                                          replace_whitespace=False, # Keep existing spaces
                                          drop_whitespace=False) # Keep trailing spaces if any
            if first_part and wrapped_lines:
                 metadata_lines.append(header + wrapped_lines[0])
                 metadata_lines.extend(wrapped_lines[1:])
                 first_part = False
            elif wrapped_lines: # Handle parts after the first explicit newline
                 metadata_lines.extend(wrapped_lines)
            elif not first_part: # Handle empty line from double newline "\n\n"
                metadata_lines.append("")


    total_lines = len(metadata_lines)
    metadata_str_processed = "\n".join(metadata_lines)

    # Calculate text block height in inches (points to inches: divide by 72)
    # Add a small buffer (e.g., 1 extra line height) for padding above/below text
    text_block_height_inches = (total_lines + 1) * (font_size / 72.0) * line_spacing

    # --- 4. Calculate Total Figure Height ---
    # Add heights and small vertical spacing between text and frames
    vertical_spacing_inches = (font_size / 72.0) * 0.5 # Half line space
    fig_height = text_block_height_inches + frame_row_height_inches + vertical_spacing_inches

    # --- 5. Create Figure and Axes using GridSpec ---
    fig = plt.figure(figsize=(fig_width, fig_height))

    # Define grid: 2 rows (text, frames), 5 columns (for frames)
    # Calculate height ratios based on calculated inches
    # Add small buffer to prevent exact edge touching
    total_content_height = text_block_height_inches + frame_row_height_inches + vertical_spacing_inches * 2
    text_height_ratio = text_block_height_inches / total_content_height
    frame_height_ratio = frame_row_height_inches / total_content_height
    spacing_ratio = (vertical_spacing_inches * 2) / total_content_height # Allocate space for margins/spacing

    # Adjust ratios to sum closer to 1, leaving room for top/bottom margins in GridSpec
    effective_h = 0.95 # Use 95% of figure height for content
    gs = fig.add_gridspec(
        nrows=2,
        ncols=5,
        height_ratios=[text_height_ratio * effective_h, frame_height_ratio * effective_h],
        width_ratios=[1, 1, 1, 1, 1],
        left=side_margin,
        right=1 - side_margin,
        bottom=spacing_ratio * 0.1, # Small bottom margin
        top=1.0 - (spacing_ratio * 0.1), # Small top margin
        wspace=0.02,  # Minimal width spacing between frame axes
        hspace=vertical_spacing_inches / fig_height # Spacing between text and frames in figure fraction
    )

    # --- 6. Add Metadata Text ---
    ax_text = fig.add_subplot(gs[0, :]) # Text spans all columns in the first row
    ax_text.text(
        0, # Start at the left edge of the text axes
        1.0, # Start at the top edge of the text axes
        metadata_str_processed,
        fontsize=font_size,
        fontfamily='monospace', # Monospace often helps with alignment/wrapping estimates
        va='top', # Vertical alignment from the top
        ha='left', # Horizontal alignment from the left
        linespacing=line_spacing,
        wrap=False # We pre-wrapped the text based on estimated width
    )
    ax_text.axis('off') # Hide axes lines and ticks for the text area

    # --- 7. Add Frames ---
    for i in range(5):
        ax_frame = fig.add_subplot(gs[1, i]) # Add frame to its column in the second row
        ax_frame.imshow(frames[i])
        ax_frame.axis('off') # Hide axes lines and ticks

    # --- 8. Save and Show ---
    try:
        # Create directory if it doesn't exist
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Figure saved to {save_path}")
    except Exception as e:
        print(f"Error saving figure: {e}")

    if show:
        plt.show()

    # Close the figure to free memory
    plt.close(fig)


In [ ]:
describe_samples_path = "logs/strokerehab_describe/llava_next_video_72b/lmms-lab__LLaVA-NeXT-Video-72B-Qwen2/20250416_103536_samples_strokerehab_describe.jsonl"

with open(describe_samples_path, "r") as f:
    data = f.readlines()
    data = [json.loads(line) for line in data]
    
    
for line in data:

    patient = line['doc']['patient']
    if patient != 'C00011':
        continue

    path_v = line['doc']['path_v']
    annotations = {
        'Patient': line['doc']['patient'],
        'Activity': line['doc']['activity'],
        'Prompt': line['input'],
        'LLM Output': line['pred'],
        'Judge Score': line['llm_score'],
        # 'Video Path': line['doc']['path_v'],
    }

    video_dir = "/gpfs/data/schambralab/quantitativeRehabilitation/__data/VideoData/rawVideosADLsandFM/"
    video_path = video_dir + path_v
    out_dir = "visualization/plots/frames"
    save_path = f"visualization/plots/{path_v.replace('.avi', '_describe.png').replace('.mkv', '_describe.png')}"

    extract_frames(video_path, 5, out_dir)
    subprocess.run(["deface", f"{out_dir}"])
    visualize_frames_with_metadata(annotations, save_path, out_dir)

In [ ]:
summarization_samples_path = "logs/strokerehab_summarization/llava_next_video_72b/lmms-lab__LLaVA-NeXT-Video-72B-Qwen2/20250415_001518_samples_strokerehab_summarization.jsonl"
with open(summarization_samples_path, "r") as f:
    data = f.readlines()
    data = [json.loads(line) for line in data]

for line in data:
    patient = line['doc']['patient']

    path_v = line['doc']['path_v']
    annotations = {
        'Patient': line['doc']['patient'],
        'Activity': line['doc']['activity'],
        'Prompt': line['input'],
        'LLM Output': line['pred'],
        'Ground Truth': line['target'],
        'Summary Steps Score': f"{float(line['summary_steps_score']):.2f}",
        # 'Video Path': line['doc']['path_v'],
    }

    video_dir = "/gpfs/data/schambralab/quantitativeRehabilitation/__data/VideoData/rawVideosADLsandFM/"
    video_path = video_dir + path_v
    out_dir = "visualization/plots/frames"
    save_path = f"visualization/plots/{path_v.replace('.avi', '_summarization.png').replace('.mkv', '_summarization.png')}"

    extract_frames(video_path, 5, out_dir)
    subprocess.run(["deface", f"{out_dir}"])
    visualize_frames_with_metadata(annotations, save_path, out_dir)

In [ ]:
primitives_samples_path = "logs/strokerehab_primitives/llava_next_video_72b/lmms-lab__LLaVA-NeXT-Video-72B-Qwen2/20250412_091958_samples_strokerehab_primitives.jsonl"
with open(primitives_samples_path, "r") as f:
    data = f.readlines()
    data = [json.loads(line) for line in data]

for line in data:

    patient = line['doc']['patient']

    path_v = line['doc']['path_v']
    maes = (
        f"{str(line['reach_mae'])} / "
        f"{str(line['reposition_mae'])} / "
        f"{str(line['transport_mae'])} / "
        f"{str(line['stabilize_mae'])} / "
        f"{str(line['idle_mae'])}"
    )
    annotations = {
        'Patient': line['doc']['patient'],
        'Activity': line['doc']['activity'],
        'Prompt': line['input'],
        'Start of LLM Output': str(line['resps'][0])[:300],
        'Start of Ground Truth': line['target'][:300],
        'Edit Score': f"{float(line['edit_score']):.2f}",
        'Action Error Rate': f"{float(line['action_error_rate']):.2f}",
        'Reach / Reposition / Transport / Stabilize / Idle MAE': maes,
        'Average MAE': f"{float(line['avg_mae']):.2f}"
    }

    video_dir = "/gpfs/data/schambralab/quantitativeRehabilitation/__data/VideoData/rawVideosADLsandFM/"
    video_path = video_dir + path_v
    out_dir = "visualization/plots/frames"
    save_path = f"visualization/plots/{path_v.replace('.avi', '_primitives.png').replace('.mkv', '_primitives.png')}"

    extract_frames(video_path, 5, out_dir)
    subprocess.run(["deface", f"{out_dir}"])
    visualize_frames_with_metadata(annotations, save_path, out_dir)